In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

In [2]:
# from google.oauth2 import service_account
# from google.cloud import bigquery

# # 빅쿼리 설정
# SERVICE_ACCOUNT_FILE = "./api_key.json"  # 키 json 파일
# credentials = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE)
# project_id = "bigquery-test-408414" # 각자 프로젝트에 맞게 수정
# client = bigquery.Client(credentials=credentials, project=project_id)

# def import_bigquery_data(query):
#     query_job = client.query(query)
#     return query_job.to_dataframe()

# events = import_bigquery_data("""
#     SELECT * 
#     FROM `bigquery-public-data.thelook_ecommerce.events` 
#     WHERE created_at BETWEEN '2024-01-01' AND '2024-09-01'
# """)

# events.to_csv('event.csv', index=False)

In [3]:
order_items = pd.read_csv('order_items.csv')
orders = pd.read_csv('orders.csv')
users = pd.read_csv('users.csv')
events = pd.read_csv('event.csv')

In [4]:
# revenue 데이터 생성
using_merge_data = pd.merge(
    order_items[~order_items['status'].isin(['Cancelled','Returned'])],
    orders,
    how='inner',
    on='order_id'
)
using_merge_data['revenue'] = round(using_merge_data['num_of_item'] * using_merge_data['sale_price'])
order_data = using_merge_data[['order_id','product_id','revenue']]

---

# 2. 퍼널 차트
- 참고 링크 : https://velog.io/@suminwooo/AARRR-%EC%A0%95%EB%A6%AC

### AARRR 개념
1. Acquisition(획득)
  - 이러한 기준은 서비스에 따라 신규 방문, 앱 설치, 서비스 결제 등으로 설정 가능
  - 신규 유저 획득을 위한 적절한 채널을 확보하는 게 중요
2. Activation(활동, 활성화)
  - 획득과 마찬가지로 서비스에 따라 상이
  - 장바구니 담기, 상세 페이지 조회, 회원 가입으로 볼 수 있음
3. Retention(유지)
  - 고객 유지가 잘 되고 있느냐를 확인하는 단계
  - 재구매율, 재방문율등 서비스에 따라 상이
4. Revenue(매출)
5. Refferal(추천)
  - 마지막 단계인 만큼 퍼널에서 가장 아래에 위치
  - 추천으로 이뤄지기 어려운 만큼 다양한 방법에 대해 고민이 필요한 단계

### 아래의 예시에서 사용할 지표
1. Acquisition(획득) : 유저 유입
2. Activation(활동, 활성화) : 회원가입
3. Retention(유지) : 재방문
4. Revenue(매출) : 구매
5. Refferal(추천) : 추천 

참고사항 : 예시는 한달 데이터만 활용(2024년 7월)

In [5]:
# 2024년 7월 가입 유저
july_usrs_lst = users[
    (users['created_at']>='2024-07-01') 
    & (users['created_at']<'2024-08-01')
]['id'].tolist()

In [6]:
events['user_id'] = events['user_id'].fillna(-1)

using_events = events[
    (events['created_at']>='2024-07-01') 
    & (events['created_at']<'2024-08-01')
    & (
        (events['user_id'] == -1) | (events['user_id'].isin(july_usrs_lst))
    )
]

### 1. Acquisition
- 유저 유입 계산
- 편의상 해당일 접속 user_id + user_id가 없는 경우엔 session_id 수

In [7]:
user_id_lst = list(set(using_events[using_events['user_id'] != -1]['user_id'].astype(int)))

# 이후 값 통일을 위해 정수형 변형 후 문자형으로
user_id_lst = [str(int(i)) for i in user_id_lst]

In [8]:
# 비회원인 경우에는 유저의 session_id를 뺀 값을 활용
user_session_lst = set(using_events[using_events['user_id'].isin(user_id_lst)]['session_id'])
not_user_session_lst = set(using_events[using_events['user_id']==-1]['session_id'])

not_user_lst = list(not_user_session_lst - user_session_lst)

In [9]:
acquisition = pd.DataFrame([user_id_lst + not_user_lst], index=['user_id']).T
acquisition['type'] = 1

In [10]:
acquisition.head()

,user_id,type
0,12290,1
1,57348,1
2,73735,1
3,38921,1
4,49163,1


### 2. Activation(활동, 활성화) : 회원가입
- 1번에서 편의상 회원은 구해두었으므로, 그대로 활용

In [11]:
activation = pd.DataFrame(user_id_lst, columns=['user_id'])
activation['type'] = 2

In [12]:
activation.head()

,user_id,type
0,12290,2
1,57348,2
2,73735,2
3,38921,2
4,49163,2


### 3. Retention
- 홈 방문 및 장바구니 기준으로 설정

In [13]:
value_counts = using_events[
    (using_events['event_type'].isin(['cart','home'])) &
    (using_events['user_id']!=-1)
]['user_id'].value_counts()
revisit_usr_lst = list(set(value_counts[value_counts >= 2].index))

# 이후 값 통일을 위해 정수형 변형 후 문자형으로
revisit_usr_lst = [str(int(i)) for i in revisit_usr_lst]

In [14]:
retention = pd.DataFrame(revisit_usr_lst, columns=['user_id'])
retention['type'] = 3

In [15]:
retention.head()

,user_id,type
0,12290,3
1,57348,3
2,73735,3
3,38921,3
4,49163,3


### 4. Revenue


In [16]:
order_usr_lst = list(set(using_events[using_events['event_type']=='purchase']['user_id']))

# 이후 값 통일을 위해 정수형 변형 후 문자형으로
order_usr_lst = [str(int(i)) for i in order_usr_lst]

In [17]:
revenue = pd.DataFrame(order_usr_lst, columns=['user_id'])
revenue['type'] = 4

### 5. Refferal
- 생략

## 퍼널 데이터 생성하기

In [18]:
# 데이터 합치기
aarrr_data = pd.concat([acquisition, activation, retention, revenue])
aarrr_data.head()

,user_id,type
0,12290,1
1,57348,1
2,73735,1
3,38921,1
4,49163,1


In [19]:
# 데이터 구성 확인
aarrr_data['type'].value_counts()

type
1    8259
2     610
3     609
4     598
Name: count, dtype: int64

In [20]:
# 퍼널 데이터 생성
phase_list = [1,2,3,4]
result_list = []

intersectArray = np.array(aarrr_data.loc[(aarrr_data['type'] == phase_list[0]), 'user_id']) #array 초기화
initialCnt = len(intersectArray) #모수

phase_name = phase_list[0] #시작 퍼널

for phase in phase_list:
    # 단계별 유저수
    phaseArray = np.array(aarrr_data.loc[(aarrr_data['type'] == phase), 'user_id']) 
    # 퍼널 통과 유저 array 
    intersectArray = set(intersectArray) & set(phaseArray) 
    intersectCnt = len(intersectArray) #퍼널 통과 유저수
    # 데이터 입력
    result_list.append([phase, intersectCnt, intersectCnt/initialCnt])

In [21]:
# 최종 퍼널 데이터
funnel_data = pd.DataFrame(data=result_list, columns = ['퍼널단계', '유저수', '전환율'])
funnel_data.head()

,퍼널단계,유저수,전환율
0,1,8259,1.000000
1,2,610,0.073859
2,3,609,0.073738
3,4,598,0.072406


In [22]:
# 스타일 적용
funnel_data.style.format(
    {'전환율':'{:,.1%}'.format}
).set_properties(
    **{'text-align': 'left'}
).bar(
    subset=['전환율'], width=100, align='left', vmin=0, vmax=1
).set_table_styles(
    [
        {"selector": "", "props": [("border", "1px solid grey")]},
        {"selector": "tbody td", "props": [("border", "1px solid grey")]},
        {"selector": "th", "props": [("border", "1px solid grey")]}]
)

,퍼널단계,유저수,전환율
0,1,8259,100.0%
1,2,610,7.4%
2,3,609,7.4%
3,4,598,7.2%
